# Novel Detections

Identifies errors that **CAED detected correctly** (true positives) but **RAHA missed**.

**Inputs required:**
- One CAED detection run (`consolidated_error_annotations.csv`)
- RAHA annotated output (`annotated_output.csv`)
- Ground truth clean + dirty datasets

**Output:**
- Count of novel detections
- `novel_detections/{dataset}/novel.csv` — cells where CAED found the error and RAHA did not

In [ ]:
import pandas as pd
import numpy as np

---
## Configuration

Set the paths to your CAED run, RAHA output, and ground truth datasets.

In [ ]:
# ── USER INPUT ───────────────────────────────────────────────────────────────

# Dataset name — used to locate ground truth files and save the output
# Options: "hospital", "beers", "flights", "rayyan"
DATASET = "flights"

# CAED run: the run ID (folder name inside backend/data/)
CAED_RUN_ID = "9afd6b23-7bcd-45d8-91ad-1fa2aef8526b"

# ── END INPUT ─────────────────────────────────────────────────────────────────

# Derived paths (change only if your layout differs)
CAED_PATH  = f"../backend/data/{CAED_RUN_ID}/consolidated_error_annotations.csv"
RAHA_PATH  = f"./tools/raha/output/{DATASET}/annotated_output.csv"
DIRTY_PATH = f"./datasets/{DATASET}/{DATASET}.csv"
CLEAN_PATH = f"./datasets/{DATASET}/clean.csv"
OUTPUT_PATH = f"./novel_detections/{DATASET}/novel.csv"

---
## Load Data

In [ ]:
caed_output   = pd.read_csv(CAED_PATH)
raha_output   = pd.read_csv(RAHA_PATH).astype(int).fillna(0)
dirty_dataset = pd.read_csv(DIRTY_PATH)
clean_dataset = pd.read_csv(CLEAN_PATH)

print(f"CAED output shape : {caed_output.shape}")
print(f"RAHA output shape : {raha_output.shape}")
print(f"Dirty dataset shape: {dirty_dataset.shape}")
print(f"Clean dataset shape: {clean_dataset.shape}")

---
## Helper Functions

In [ ]:
def annotate_errors(df_clean: pd.DataFrame, df_dirty: pd.DataFrame) -> pd.DataFrame:
    """Returns a binary matrix: 1 where clean and dirty differ (actual errors)."""
    if df_clean.shape != df_dirty.shape or not all(df_clean.columns == df_dirty.columns):
        raise ValueError("Clean and dirty datasets must have the same structure.")
    return (df_clean.astype(str) != df_dirty.astype(str)).astype(int)


def inspect_classification(
    true_dataset: pd.DataFrame,
    pred_dataset: pd.DataFrame,
    input_dataset: pd.DataFrame,
):
    """
    Classifies predictions into TP, FP, FN relative to ground truth.
    Returns DataFrames containing the actual cell values (from input_dataset)
    for each category, with 0 elsewhere.
    """
    true_dataset = true_dataset.reset_index(drop=True).copy()
    pred_dataset = pred_dataset.reset_index(drop=True).copy()
    input_dataset = input_dataset.reset_index(drop=True).copy()

    true_dataset.columns = input_dataset.columns
    pred_dataset.columns = input_dataset.columns

    # Encoding: true=1 → +2, pred=1 → +1, pred=0 → -1
    # TP: 2+1 = 3+1 = 4  (true=1, pred=1)
    # FP: 2+(-1) = 2  ... wait, let me re-check the original logic:
    # calc = true.add(2): true=0 → 2, true=1 → 3
    # calc_out: pred=0 → -1, pred=1 → 1
    # calc + calc_out: TP = 3+1=4, FP = 2+1=3, FN = 3+(-1)=2, TN = 2+(-1)=1
    calc = true_dataset.add(2)
    calc_out = pred_dataset.copy()
    calc_out[calc_out == 0] = -1
    calc = calc.add(calc_out)

    def extract(mask):
        df = input_dataset[mask].astype(str).replace("nan", 0)
        return df.reset_index(drop=True)

    tp_df  = extract(calc == 4)
    fp_df  = extract(calc == 3)
    fn_df  = extract(calc == 2)

    return tp_df, fp_df, fn_df


def is_non_zero(value):
    """Returns True if value is non-zero. Handles strings that can't be cast to float."""
    try:
        return float(value) != 0.0
    except (ValueError, TypeError):
        return True  # Non-numeric strings are treated as non-zero

---
## Compute Novel Detections

In [ ]:
# Ground truth error locations
error_annotation = annotate_errors(clean_dataset, dirty_dataset)

# CAED true positives
caed_tp, _, _ = inspect_classification(error_annotation, caed_output, dirty_dataset)

# RAHA true positives
raha_tp, _, _ = inspect_classification(error_annotation, raha_output, dirty_dataset)

print(f"CAED TP cell count : {caed_tp.applymap(is_non_zero).sum().sum()}")
print(f"RAHA TP cell count : {raha_tp.applymap(is_non_zero).sum().sum()}")

In [ ]:
# Novel detections: cells where CAED found the error but RAHA did not
difference_mask = caed_tp != raha_tp
novel = caed_tp.where(difference_mask, np.nan).fillna(0)

non_zero_mask = novel.applymap(is_non_zero)
novel_count = non_zero_mask.sum().sum()

print(f"Novel detections (CAED TP, RAHA missed): {novel_count}")

---
## Results

In [ ]:
# Preview rows that contain at least one novel detection
rows_with_novel = non_zero_mask.any(axis=1)
print(f"Rows with novel detections: {rows_with_novel.sum()}")
novel[rows_with_novel]

In [ ]:
# Novel detections broken down by column
column_counts = non_zero_mask.sum().rename("novel_detections")
column_counts[column_counts > 0].sort_values(ascending=False)

In [ ]:
# Save to novel_detections/{dataset}/novel.csv
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
novel.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")